# Phase 4i: Equal-Class Undersampling on the Corrected v6 Set (v8)

**Single-variable test, continuing from `skin_cancer_phase4g.ipynb` (v6, the
current 7-class best -- macro AUC 0.8789, melanoma AUC 0.7927).** The only
change vs. Phase 4g is the **training-set class balance**:

- `data/train_v8.csv`, built by `_build_train_v8_balanced.py` (seed=42):
  every one of the 7 v6 classes is undersampled to the minority-class size
  (dermatofibroma = 74), giving **518 train images total** (vs 4,624 in
  `train_v6.csv`, which kept nevus capped at 3,000 but left every other class
  at its natural count). `val_v6.csv` / `test_v6.csv` are **untouched** (same
  splits, same corrected 7-class labels, no leakage from balancing).
- `CLASS_WEIGHTS` recomputed (sqrt-balanced) on `train_v8.csv` -- since the
  balanced set is perfectly uniform this comes out ~1.0 for every class, i.e.
  the loss is effectively unweighted focal loss (the imbalance correction now
  lives entirely in the *data*, not the loss).

Everything else is kept exactly as v6/v4 (per the user's explicit
instruction -- **no EMA, no CBAM, no Balanced Softmax, no architectural
changes**):

- Same architecture: EfficientNet-B4 + SE block (`IN_FEATURES=1792`)
- Same loss: `FocalLoss(alpha=CLASS_WEIGHTS, gamma=2.0, label_smoothing=0.1)`
- Same sampler (plain `shuffle=True`)
- Same optimizer/scheduler: AdamW + LinearLR warmup -> CosineAnnealingWarmRestarts
- Same 3-stage training structure (10/20/10 epochs, lrs 1e-3/1e-4/1e-5),
  `EARLY_STOP_PATIENCE=8`
- Same checkpoint criterion (best `val_auc_macro`)
- Same augmentation pipeline (v2: multi-crop, wider rotation, affine zoom,
  color jitter), same resolution (380x380, `preprocessed_v6_resolution380`
  via `preprocessing_config_v6.json`)
- Same memory adaptation as v4/v6/380px: micro-batch=16, grad-accum=2
  (effective batch 32)
- Same starting weights (`model_init.pth`, Phase 3 backbone, loaded with
  shape-mismatch-safe filtering for the 256->7 classifier layer)

`MALIGNANT_CLASSES = ['melanoma', 'basal cell carcinoma']` (v6 7-class
taxonomy -- squamous cell carcinoma stays removed).

Outputs are tagged `_v8` (e.g. `best_model_stage{N}_v8.pth`,
`final_model_v8.pth`, `history_stage{N}_v8.csv`, `training_curves_stage{N}_v8.png`)
so v6 and all prior-attempt artifacts are preserved untouched.

In [ ]:
import json, logging, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BASE_DIR    = Path(r'c:\graduation project')
DATA_DIR    = BASE_DIR / 'data'
MODELS_DIR  = BASE_DIR / 'models' / 'checkpoints'
METRICS_DIR = BASE_DIR / 'results' / 'metrics'
FIGURES_DIR = BASE_DIR / 'results' / 'figures'

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('phase4i')

# Phase 4i -- same v6 config (classes, normalization, resolution, preprocessed
# directory) as Phase 4g. CLASS_WEIGHTS is NOT taken from cfg_v6 here -- it is
# recomputed below (cell 3) from the equal-class-undersampled train_v8 set.
with open(DATA_DIR / 'preprocessing_config_v6.json') as f:
    cfg_v6 = json.load(f)

CLASSES      = cfg_v6['classes']
NUM_CLASSES  = cfg_v6['num_classes']
NORM_MEAN    = cfg_v6['norm_mean']
NORM_STD     = cfg_v6['norm_std']
MALIGNANT_CLASSES = ['melanoma', 'basal cell carcinoma']

IMG_SIZE = cfg_v6['img_size']                                    # 380 (same as v4/v6)
V6_DIR   = cfg_v6['preprocessed_dir']                            # 'preprocessed_v6_resolution380'

# Memory adaptation: identical to v4/v6/380px (micro-batch=16, accum=2 -> effective 32)
MICRO_BATCH          = 16
GRAD_ACCUM_STEPS     = 2
BATCH_SIZE           = MICRO_BATCH
EFFECTIVE_BATCH_SIZE = MICRO_BATCH * GRAD_ACCUM_STEPS            # == cfg_v6['batch_size'] (32)

print(f"Device          : {DEVICE}")
print(f"Classes ({NUM_CLASSES})     : {CLASSES}")
print(f"Resolution      : {IMG_SIZE}x{IMG_SIZE}")
print(f"Micro batch     : {MICRO_BATCH}")
print(f"Grad accum steps: {GRAD_ACCUM_STEPS}")
print(f"Effective batch : {EFFECTIVE_BATCH_SIZE}")
print(f"AMP             : enabled" if DEVICE.type == 'cuda' else "AMP: disabled (CPU)")

## 1 - DataLoaders (train_v8.csv -- equal-class undersampled, 74/class)

In [ ]:
class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        # v6 metadata already points at the 380px preprocessed images
        path  = row['preprocessed_path_v6']
        image = Image.open(path).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, int(row['label_int'])


# v2 augmentation pipeline (unchanged from baseline/v4/v6): wider rotation
# (+/-20), true zoom in/out (0.8x-1.2x via affine), brightness/contrast +/-10%,
# plus "multi-crop" -- each sample randomly gets either the full lesion view
# or a tighter zoomed-in crop (RandomResizedCrop 0.5-0.9).
multi_crop_v2 = T.RandomChoice([
    T.Resize((IMG_SIZE, IMG_SIZE)),                                     # full lesion view
    T.RandomResizedCrop(IMG_SIZE, scale=(0.5, 0.9), ratio=(0.9, 1.1)),  # zoomed lesion crop
])
train_transform = T.Compose([
    multi_crop_v2,
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.3),
    T.RandomRotation(degrees=20),
    T.RandomAffine(degrees=0, scale=(0.8, 1.2)),
    T.ColorJitter(brightness=0.10, contrast=0.10),
    T.ToTensor(),
    T.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

# train_v8.csv: equal-class undersampled (74/class, seed=42, minority=dermatofibroma).
# val_v6.csv: untouched (same v6 split, no leakage from balancing).
train_df = pd.read_csv(DATA_DIR / 'train_v8.csv')
val_df   = pd.read_csv(DATA_DIR / 'val_v6.csv')

train_loader = DataLoader(SkinLesionDataset(train_df, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(SkinLesionDataset(val_df,   val_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

log.info(f"Train: {len(train_df):,} ({len(train_loader)} batches)  |  Val: {len(val_df):,} ({len(val_loader)} batches)")
log.info("Train class counts:\n" + train_df['label_str'].value_counts().reindex(CLASSES).to_string())

# Class weights recomputed on the balanced train_v8 set (sqrt-smoothed
# 'balanced' weighting, same formula as v1/v4/v6/v7). With perfectly equal
# class counts this comes out ~1.0 for every class -- the loss is effectively
# unweighted focal loss, since the imbalance correction now lives in the data.
class_weights_arr = np.sqrt(compute_class_weight(
    'balanced', classes=np.arange(NUM_CLASSES), y=train_df['label_int'].values))
CLASS_WEIGHTS = torch.tensor(class_weights_arr, dtype=torch.float32).to(DEVICE)
log.info("Class weights (sqrt-balanced, train_v8): " +
         str({CLASSES[i]: round(float(class_weights_arr[i]), 4) for i in range(NUM_CLASSES)}))

## 2 - Model

In [ ]:
# Squeeze-and-Excitation block (Hu et al., 2018).
# Learns per-channel attention weights from global context so the model can
# emphasise the feature channels most relevant to a given lesion
# (color/texture/border cues) before pooling into the classifier.
class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class SkinCancerModel(nn.Module):
    IN_FEATURES = 1792

    def __init__(self, num_classes: int = 8):
        super().__init__()
        base = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
        self.features = base.features
        self.se       = SEBlock(self.IN_FEATURES, reduction=16)
        self.avgpool  = base.avgpool
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(self.IN_FEATURES),
            nn.Linear(self.IN_FEATURES, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes),
        )
        self.set_stage(1)

    def set_stage(self, stage: int):
        for p in self.features.parameters():
            p.requires_grad = False
        if stage == 2:
            for block in list(self.features.children())[5:]:
                for p in block.parameters():
                    p.requires_grad = True
        elif stage == 3:
            for p in self.features.parameters():
                p.requires_grad = True
        for p in self.se.parameters():
            p.requires_grad = True
        for p in self.classifier.parameters():
            p.requires_grad = True

    def forward(self, x):
        x = self.features(x)
        x = self.se(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


# Multi-class focal loss (Lin et al., 2017).
# Down-weights easy/well-classified examples via (1 - p_t)^gamma so the model
# spends more learning capacity on hard/rare cases, while `alpha`
# (CLASS_WEIGHTS) provides additional per-class balancing. With train_v8
# perfectly balanced, CLASS_WEIGHTS ~= 1.0 for every class.
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma: float = 2.0, label_smoothing: float = 0.1):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha,
                              label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-ce)
        focal = (1 - pt) ** self.gamma * ce
        return focal.mean()


model = SkinCancerModel(num_classes=NUM_CLASSES).to(DEVICE)

# Same starting point as the baseline/v4/v6 (Phase 3 backbone weights). The
# 8-class taxonomy is now 7-class, so the final classifier layer
# (classifier.7: Linear(256, 8) -> Linear(256, 7)) has a shape mismatch.
# Filter out any tensors whose shape doesn't match the current model before
# calling load_state_dict(strict=False) -- everything else (backbone, SE
# block, earlier classifier layers) loads normally and is fine-tuned from
# the same starting point as every prior run.
init_path = MODELS_DIR / 'model_init.pth'
if init_path.exists():
    init_state  = torch.load(init_path, map_location=DEVICE, weights_only=True)
    model_state = model.state_dict()
    compatible  = {k: v for k, v in init_state.items()
                    if k in model_state and v.shape == model_state[k].shape}
    skipped = sorted(set(init_state) - set(compatible))
    missing, unexpected = model.load_state_dict(compatible, strict=False)
    log.info(f"Loaded initial weights from Phase 3 (skipped shape-mismatch={skipped}, "
             f"missing={list(missing)})")

criterion = FocalLoss(alpha=CLASS_WEIGHTS, gamma=2.0, label_smoothing=0.1)

STAGE_CFG = {
    1: {'epochs': 10, 'lr': 1e-3, 'desc': 'Head only'},
    2: {'epochs': 20, 'lr': 1e-4, 'desc': 'Head + last 4 backbone blocks'},
    3: {'epochs': 10, 'lr': 1e-5, 'desc': 'Full model'},
}
EARLY_STOP_PATIENCE = 8

print("Model loaded ✓")

## 3 - Medical Metrics

In [ ]:
# Compute macro AUC, and per-class sensitivity/specificity (one-vs-rest).
# Sensitivity = recall for that class (TP / (TP+FN))
# Specificity = true negative rate (TN / (TN+FP))
def compute_medical_metrics(y_true: np.ndarray, y_proba: np.ndarray) -> dict:
    y_pred = y_proba.argmax(axis=1)

    try:
        auc_macro = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')
    except ValueError:
        auc_macro = float('nan')

    cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)))
    sens, spec = {}, {}
    for i, cls in enumerate(CLASSES):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        sens[cls] = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
        spec[cls] = tn / (tn + fp) if (tn + fp) > 0 else float('nan')

    return {'auc_macro': auc_macro, 'sensitivity': sens, 'specificity': spec, 'confusion_matrix': cm}


print("compute_medical_metrics defined ✓")

## 4 - Train / Eval Epoch Functions (AMP-enabled, gradient accumulation)

In [ ]:
# Same training step as the baseline/v4/v6, generalized with gradient
# accumulation. For accum_steps=1 this is identical to the baseline's
# train_one_epoch.
def train_one_epoch(model, loader, criterion, optimizer, scaler, device, accum_steps=1):
    model.train()
    running_loss = 0.0
    use_amp = device.type == 'cuda'
    n_batches = len(loader)
    optimizer.zero_grad(set_to_none=True)

    for i, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)

        with autocast(device_type=device.type, enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels) / accum_steps

        scaler.scale(loss).backward()

        if (i + 1) % accum_steps == 0 or (i + 1) == n_batches:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * images.size(0) * accum_steps

    return running_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_labels, all_probs = [], []
    use_amp = device.type == 'cuda'

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        with autocast(device_type=device.type, enabled=use_amp):
            outputs = model(images)
            loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        all_probs.append(torch.softmax(outputs.float(), dim=1).cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    val_loss = running_loss / len(loader.dataset)
    y_true  = np.concatenate(all_labels)
    y_proba = np.concatenate(all_probs)
    metrics = compute_medical_metrics(y_true, y_proba)
    metrics['loss'] = val_loss
    return metrics


print("train_one_epoch / evaluate defined ✓")

## 5 - Optimiser / Scheduler Builders

In [ ]:
def build_optimizer(model, stage):
    base_lr = STAGE_CFG[stage]['lr']
    if stage == 1:
        return torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=base_lr, weight_decay=1e-4
        )
    head_params     = list(model.classifier.parameters()) + list(model.se.parameters())
    backbone_params = [p for p in model.features.parameters() if p.requires_grad]
    return torch.optim.AdamW([
        {'params': head_params,     'lr': base_lr},
        {'params': backbone_params, 'lr': base_lr / 10},
    ], weight_decay=1e-4)


# Linear warmup for the first ~10% of epochs, then cosine annealing with warm
# restarts. Restarting the LR cycle periodically helps the optimizer escape
# the loss plateaus seen with ReduceLROnPlateau in earlier runs.
def build_scheduler(optimizer, epochs):
    warmup_epochs = max(1, epochs // 10)
    warmup = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, total_iters=warmup_epochs)
    cosine = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=max(2, (epochs - warmup_epochs) // 2), T_mult=1, eta_min=1e-7)
    return torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup, cosine], milestones=[warmup_epochs])


print("Optimiser/scheduler builders defined ✓")

## 6 - Training Curves Plot

In [ ]:
def plot_training_curves(history_df: pd.DataFrame, stage: int):
    best_idx = history_df['val_auc_macro'].idxmax()
    best_epoch = history_df.loc[best_idx, 'epoch']

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(history_df['epoch'], history_df['train_loss'], label='train', marker='o', ms=3)
    axes[0].plot(history_df['epoch'], history_df['val_loss'],   label='val',   marker='o', ms=3)
    axes[0].axvline(best_epoch, color='red', linestyle='--', alpha=0.6, label='best epoch')
    axes[0].set_title(f'Stage {stage} (v8, equal-class 380px) - Loss')
    axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(history_df['epoch'], history_df['val_auc_macro'], color='#1D9E75', marker='o', ms=3)
    axes[1].axvline(best_epoch, color='red', linestyle='--', alpha=0.6)
    axes[1].axhline(0.93, color='gray', linestyle=':', label='target 0.93')
    axes[1].set_title(f'Stage {stage} (v8, equal-class 380px) - Val Macro AUC')
    axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(history_df['epoch'], history_df['sens_melanoma'], color='#E24B4A', marker='o', ms=3, label='Melanoma')
    if 'sens_basal_cell_carcinoma' in history_df:
        axes[2].plot(history_df['epoch'], history_df['sens_basal_cell_carcinoma'], color='#D85A30', marker='o', ms=3, label='BCC')
    axes[2].axhline(0.85, color='gray', linestyle=':', label='target 0.85')
    axes[2].axvline(best_epoch, color='red', linestyle='--', alpha=0.6)
    axes[2].set_title(f'Stage {stage} (v8, equal-class 380px) - Malignant Sensitivity')
    axes[2].set_xlabel('Epoch'); axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

    plt.suptitle(f'Stage {stage} Training Curves (Phase 4i -- v8 equal-class undersampling)', fontweight='bold')
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / f'training_curves_stage{stage}_v8.png', dpi=150, bbox_inches='tight')
    plt.show()


print("plot_training_curves defined ✓")

## 7 - Stage Training Loop (checkpoint + resume)

In [ ]:
def train_stage(model, stage, train_loader, val_loader, criterion, device):
    epochs = STAGE_CFG[stage]['epochs']
    model.set_stage(stage)

    optimizer = build_optimizer(model, stage)
    scheduler = build_scheduler(optimizer, epochs)
    scaler    = GradScaler(device.type, enabled=(device.type == 'cuda'))

    checkpoint_path = MODELS_DIR / f'best_model_stage{stage}_v8.pth'
    history_path    = METRICS_DIR / f'history_stage{stage}_v8.csv'
    state_path      = MODELS_DIR / 'training_state_v8.json'

    best_val_auc = -1.0
    start_epoch  = 0
    history      = []
    patience_ctr = 0

    # -- Resume logic --------------------------------------------------------
    if state_path.exists():
        state = json.loads(state_path.read_text())
        if state.get('stage') == stage:
            start_epoch  = state['last_epoch'] + 1
            best_val_auc = state['best_val_auc']
            if checkpoint_path.exists():
                model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
            if history_path.exists():
                history = pd.read_csv(history_path).to_dict('records')
            print(f"Resuming Stage {stage} from epoch {start_epoch}, best AUC = {best_val_auc:.4f}")

    if start_epoch >= epochs:
        print(f"Stage {stage} already complete ({start_epoch}/{epochs} epochs).")
        if checkpoint_path.exists():
            model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
        return pd.DataFrame(history), best_val_auc

    print(f"\n{'='*70}\n  STAGE {stage}: {STAGE_CFG[stage]['desc']}  "
          f"({epochs} epochs, lr={STAGE_CFG[stage]['lr']})\n{'='*70}")

    for epoch in range(start_epoch, epochs):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device,
                                      accum_steps=GRAD_ACCUM_STEPS)
        val_metrics = evaluate(model, val_loader, criterion, device)
        scheduler.step()
        elapsed = time.time() - t0

        row = {
            'epoch'     : epoch,
            'train_loss': train_loss,
            'val_loss'  : val_metrics['loss'],
            'val_auc_macro': val_metrics['auc_macro'],
            'lr'        : optimizer.param_groups[0]['lr'],
            'time_sec'  : round(elapsed, 1),
        }
        for cls in MALIGNANT_CLASSES:
            key = cls.replace(' ', '_')
            row[f'sens_{key}'] = val_metrics['sensitivity'][cls]
            row[f'spec_{key}'] = val_metrics['specificity'][cls]

        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)

        print(f"  Epoch {epoch+1:>2}/{epochs} | loss={train_loss:.4f} "
              f"val_loss={val_metrics['loss']:.4f} val_auc={val_metrics['auc_macro']:.4f} "
              f"melanoma_sens={row['sens_melanoma']:.4f} | lr={row['lr']:.2e} | {elapsed:.0f}s")

        # Checkpoint best model (by val_auc)
        if val_metrics['auc_macro'] > best_val_auc:
            best_val_auc = val_metrics['auc_macro']
            torch.save(model.state_dict(), checkpoint_path)
            patience_ctr = 0
            print(f"    -> new best (val_auc={best_val_auc:.4f}) -- checkpoint saved")
        else:
            patience_ctr += 1

        state_path.write_text(json.dumps({
            'stage': stage, 'last_epoch': epoch, 'best_val_auc': best_val_auc
        }))

        if patience_ctr >= EARLY_STOP_PATIENCE:
            print(f"  Early stopping -- no improvement for {EARLY_STOP_PATIENCE} epochs")
            break

    # Restore best weights for this stage
    model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    history_df = pd.DataFrame(history)
    plot_training_curves(history_df, stage)
    return history_df, best_val_auc


print("train_stage defined ✓")

## 8 - Stage 1 - Train Head Only

In [ ]:
history_s1, best_auc_s1 = train_stage(model, 1, train_loader, val_loader, criterion, DEVICE)
print(f"\nStage 1 best val AUC: {best_auc_s1:.4f}")
torch.cuda.empty_cache()

## 9 - Stage 2 - Unfreeze Last 4 Backbone Blocks
Proceeds only if Stage 1 achieved val_auc > 0.75

In [ ]:
if best_auc_s1 > 0.75:
    history_s2, best_auc_s2 = train_stage(model, 2, train_loader, val_loader, criterion, DEVICE)
    print(f"\nStage 2 best val AUC: {best_auc_s2:.4f}")
else:
    print(f"Stage 1 val_auc ({best_auc_s1:.4f}) <= 0.75 -- skipping Stage 2.")
    print("Consider: more epochs in Stage 1, or check data/preprocessing quality.")
    history_s2, best_auc_s2 = pd.DataFrame(), best_auc_s1

torch.cuda.empty_cache()

## 10 - Stage 3 - Full Fine-Tuning
Proceeds only if Stage 2 improved val_auc by > 1%

In [ ]:
improvement = best_auc_s2 - best_auc_s1

if not history_s2.empty and improvement > 0.01:
    history_s3, best_auc_s3 = train_stage(model, 3, train_loader, val_loader, criterion, DEVICE)
    print(f"\nStage 3 best val AUC: {best_auc_s3:.4f}")
else:
    print(f"Stage 2->2 improvement ({improvement:.4f}) <= 0.01 -- skipping Stage 3.")
    history_s3, best_auc_s3 = pd.DataFrame(), best_auc_s2

torch.cuda.empty_cache()

## 11 - Final Model

In [ ]:
# Save the final best model (whichever stage achieved highest val_auc)
best_overall = max(
    [(best_auc_s1, 1), (best_auc_s2, 2), (best_auc_s3, 3)],
    key=lambda x: x[0]
)
best_auc, best_stage = best_overall
best_ckpt = MODELS_DIR / f'best_model_stage{best_stage}_v8.pth'

model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE, weights_only=True))
final_path = MODELS_DIR / 'final_model_v8.pth'
torch.save(model.state_dict(), final_path)

print("="*60)
print("  TRAINING COMPLETE (Phase 4i -- v8 equal-class undersampling on v6)")
print("="*60)
print(f"  Best stage    : {best_stage}")
print(f"  Best val AUC  : {best_auc:.4f}")
print(f"  Final model   : {final_path}")
print("="*60)

# Combined training summary
summary = pd.DataFrame([
    {'stage': 1, 'best_val_auc': best_auc_s1, 'epochs_run': len(history_s1)},
    {'stage': 2, 'best_val_auc': best_auc_s2, 'epochs_run': len(history_s2)},
    {'stage': 3, 'best_val_auc': best_auc_s3, 'epochs_run': len(history_s3)},
])
summary.to_csv(METRICS_DIR / 'training_summary_v8.csv', index=False)
print(summary.to_string(index=False))

## 12 - Evaluation Setup (5-way TTA, val_v6 + test_v6)

Evaluate `final_model_v8.pth` (already held in `model` from Cell 23) on the
**untouched** `val_v6.csv` / `test_v6.csv` splits using the same 5-way TTA
pipeline (original, horizontal flip, vertical flip, +/-10 deg rotation) as
v6/v7.

In [ ]:
import seaborn as sns
from sklearn.metrics import (roc_curve, auc, precision_recall_fscore_support,
                              cohen_kappa_score, f1_score, balanced_accuracy_score,
                              matthews_corrcoef)
from sklearn.preprocessing import label_binarize

model.eval()

tta_transforms = {
    'original'  : T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(mean=NORM_MEAN, std=NORM_STD)]),
    'h_flip'    : T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomHorizontalFlip(p=1.0), T.ToTensor(), T.Normalize(mean=NORM_MEAN, std=NORM_STD)]),
    'v_flip'    : T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomVerticalFlip(p=1.0), T.ToTensor(), T.Normalize(mean=NORM_MEAN, std=NORM_STD)]),
    'rotate_p10': T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomRotation((10, 10)), T.ToTensor(), T.Normalize(mean=NORM_MEAN, std=NORM_STD)]),
    'rotate_n10': T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomRotation((-10, -10)), T.ToTensor(), T.Normalize(mean=NORM_MEAN, std=NORM_STD)]),
}


@torch.no_grad()
def predict(model, loader, device):
    all_labels, all_probs = [], []
    use_amp = device.type == 'cuda'
    for images, labels in loader:
        images = images.to(device)
        with autocast(device_type=device.type, enabled=use_amp):
            outputs = model(images)
        probs = torch.softmax(outputs.float(), dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())
    return np.concatenate(all_labels), np.concatenate(all_probs)


def run_tta(model, loaders):
    tta_probas = []
    for name, loader in loaders.items():
        y_true, y_proba_variant = predict(model, loader, DEVICE)
        tta_probas.append(y_proba_variant)
    return y_true, np.mean(tta_probas, axis=0)


print(f"TTA variants: {list(tta_transforms.keys())}")
print("Evaluation helpers (predict / run_tta) defined ✓")

## 13 - Full Evaluation Function

Computes: accuracy, Cohen's kappa, confusion matrix (counts + row-normalized
heatmap), per-class ROC/AUC, precision/recall/F1/specificity per class, macro
F1, balanced accuracy, MCC, bootstrap 95% CIs (1000 resamples, seed=42),
Youden-threshold tuning + malignancy-prioritized threshold-adjusted decision
rule, comparison-vs-targets table, and the pigmented-lesion confusion cluster
(melanoma -> nevus / pigmented benign keratosis).

In [ ]:
def run_full_eval(df, tag, title):
    loaders = {
        name: DataLoader(SkinLesionDataset(df, tfm), batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=True)
        for name, tfm in tta_transforms.items()
    }
    print(f"\n{'='*70}\n  {title}  ({len(df):,} images, {len(next(iter(loaders.values())))} batches/TTA-variant)\n{'='*70}")

    y_true, y_proba = run_tta(model, loaders)
    y_pred = y_proba.argmax(axis=1)

    pred_df = df.copy()
    pred_df['pred_int'] = y_pred
    pred_df['pred_str'] = [CLASSES[i] for i in y_pred]
    for i, cls in enumerate(CLASSES):
        pred_df[f'proba_{cls.replace(" ", "_")}'] = y_proba[:, i]
    pred_df.to_csv(METRICS_DIR / f'{tag}_predictions.csv', index=False)

    accuracy = (y_pred == y_true).mean()
    kappa    = cohen_kappa_score(y_true, y_pred)
    print(f"Overall accuracy : {accuracy:.4f}")
    print(f"Cohen's kappa    : {kappa:.4f}")

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
    axes[0].set_title(f'Confusion Matrix (counts) - {title}')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
    axes[1].set_title(f'Confusion Matrix (row-normalized) - {title}')
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

    for ax in axes:
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
        plt.setp(ax.get_yticklabels(), rotation=0)

    plt.tight_layout()
    fig.savefig(FIGURES_DIR / f'confusion_matrix_{tag}.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ROC curves & AUC
    y_true_bin = label_binarize(y_true, classes=range(NUM_CLASSES))

    fig, ax = plt.subplots(figsize=(8, 8))
    auc_per_class = {}
    for i, cls in enumerate(CLASSES):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
        roc_auc = auc(fpr, tpr)
        auc_per_class[cls] = roc_auc
        style = '-' if cls in MALIGNANT_CLASSES else '--'
        ax.plot(fpr, tpr, style, label=f'{cls} (AUC={roc_auc:.3f})')

    macro_auc = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')
    ax.plot([0, 1], [0, 1], 'k:', alpha=0.4)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curves - {title} (Macro AUC = {macro_auc:.4f})')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(alpha=0.3)
    fig.savefig(FIGURES_DIR / f'roc_curves_{tag}.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\nMacro AUC: {macro_auc:.4f}\n")
    for cls, a in auc_per_class.items():
        marker = '  (malignant)' if cls in MALIGNANT_CLASSES else ''
        print(f"  {cls:30s}: {a:.4f}{marker}")

    # Per-class sensitivity / specificity / precision / F1
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=range(NUM_CLASSES), zero_division=0)

    specificity = []
    for i in range(NUM_CLASSES):
        tn = cm.sum() - cm[i, :].sum() - cm[:, i].sum() + cm[i, i]
        fp = cm[:, i].sum() - cm[i, i]
        specificity.append(tn / (tn + fp) if (tn + fp) > 0 else float('nan'))

    metrics_df = pd.DataFrame({
        'class': CLASSES,
        'malignant': [c in MALIGNANT_CLASSES for c in CLASSES],
        'support': support,
        'sensitivity_recall': recall,
        'specificity': specificity,
        'precision': precision,
        'f1': f1,
        'auc': [auc_per_class[c] for c in CLASSES],
    })
    metrics_df.to_csv(METRICS_DIR / f'{tag}_per_class_metrics.csv', index=False)
    print("\nPer-class metrics:")
    print(metrics_df.round(4).to_string(index=False))

    macro_f1 = f1_score(y_true, y_pred, labels=range(NUM_CLASSES), average='macro', zero_division=0)
    bal_acc  = balanced_accuracy_score(y_true, y_pred)
    mcc      = matthews_corrcoef(y_true, y_pred)
    print(f"\nMacro F1            : {macro_f1:.4f}")
    print(f"Balanced accuracy   : {bal_acc:.4f}")
    print(f"MCC                 : {mcc:.4f}")

    # Bootstrapped 95% CIs
    N_BOOTSTRAP = 1000
    rng = np.random.default_rng(SEED)
    melanoma_idx = CLASSES.index('melanoma')
    n = len(y_true)

    boot_macro_auc, boot_melanoma_sens = [], []
    for _ in range(N_BOOTSTRAP):
        idx = rng.integers(0, n, n)
        yt, yp = y_true[idx], y_proba[idx]
        try:
            boot_macro_auc.append(roc_auc_score(yt, yp, multi_class='ovr', average='macro'))
        except ValueError:
            continue
        y_pred_b = yp.argmax(axis=1)
        cm_b = confusion_matrix(yt, y_pred_b, labels=range(NUM_CLASSES))
        tp = cm_b[melanoma_idx, melanoma_idx]
        fn = cm_b[melanoma_idx, :].sum() - tp
        boot_melanoma_sens.append(tp / (tp + fn) if (tp + fn) > 0 else np.nan)

    def ci(arr, lo=2.5, hi=97.5):
        arr = np.array(arr, dtype=float)
        arr = arr[~np.isnan(arr)]
        return np.percentile(arr, lo), np.percentile(arr, hi)

    macro_lo, macro_hi = ci(boot_macro_auc)
    mel_lo, mel_hi = ci(boot_melanoma_sens)

    print(f"\nMacro AUC            : {macro_auc:.4f}  (95% CI: {macro_lo:.4f} - {macro_hi:.4f})")
    print(f"Melanoma sensitivity : {recall[melanoma_idx]:.4f}  (95% CI: {mel_lo:.4f} - {mel_hi:.4f})")

    bootstrap_summary = pd.DataFrame([
        {'metric': 'macro_auc',            'point_estimate': macro_auc,           'ci_lower': macro_lo, 'ci_upper': macro_hi},
        {'metric': 'melanoma_sensitivity', 'point_estimate': recall[melanoma_idx], 'ci_lower': mel_lo,   'ci_upper': mel_hi},
    ])
    bootstrap_summary.to_csv(METRICS_DIR / f'{tag}_bootstrap_ci.csv', index=False)

    # Decision threshold tuning for malignant classes
    threshold_results = []
    fig, axes = plt.subplots(1, len(MALIGNANT_CLASSES), figsize=(15, 4.5))

    for ax, cls in zip(axes, MALIGNANT_CLASSES):
        i = CLASSES.index(cls)
        fpr, tpr, thresholds = roc_curve(y_true_bin[:, i], y_proba[:, i])
        youden = tpr - fpr
        best_idx = youden.argmax()
        best_thresh = thresholds[best_idx]

        ax.plot(fpr, tpr, label='ROC')
        ax.scatter(fpr[best_idx], tpr[best_idx], color='red', zorder=5,
                   label=f'Youden optimal\n(thr={best_thresh:.3f})')
        ax.plot([0, 1], [0, 1], 'k:', alpha=0.3)
        ax.set_title(cls)
        ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

        threshold_results.append({
            'class': cls,
            'argmax_sensitivity': recall[i],
            'argmax_specificity': specificity[i],
            'youden_threshold': best_thresh,
            'youden_sensitivity': tpr[best_idx],
            'youden_specificity': 1 - fpr[best_idx],
        })

    plt.tight_layout()
    fig.savefig(FIGURES_DIR / f'threshold_tuning_malignant_{tag}.png', dpi=150, bbox_inches='tight')
    plt.show()

    threshold_df = pd.DataFrame(threshold_results)
    threshold_df.to_csv(METRICS_DIR / f'{tag}_threshold_tuning.csv', index=False)
    print("\nThreshold tuning:")
    print(threshold_df.round(4).to_string(index=False))

    # Malignancy-prioritized decision rule
    malignant_idx    = {cls: CLASSES.index(cls) for cls in MALIGNANT_CLASSES}
    malignant_thresh = dict(zip(threshold_df['class'], threshold_df['youden_threshold']))

    y_pred_adj = y_pred.copy()
    for i in range(len(y_proba)):
        candidates = [
            (cls, y_proba[i, malignant_idx[cls]])
            for cls in MALIGNANT_CLASSES
            if y_proba[i, malignant_idx[cls]] >= malignant_thresh[cls]
        ]
        if candidates:
            best_cls = max(candidates, key=lambda x: x[1])[0]
            y_pred_adj[i] = malignant_idx[best_cls]

    precision_adj, recall_adj, f1_adj, _ = precision_recall_fscore_support(
        y_true, y_pred_adj, labels=range(NUM_CLASSES), zero_division=0
    )
    cm_adj = confusion_matrix(y_true, y_pred_adj, labels=range(NUM_CLASSES))
    specificity_adj = []
    for i in range(NUM_CLASSES):
        tn = cm_adj.sum() - cm_adj[i, :].sum() - cm_adj[:, i].sum() + cm_adj[i, i]
        fp = cm_adj[:, i].sum() - cm_adj[i, i]
        specificity_adj.append(tn / (tn + fp) if (tn + fp) > 0 else float('nan'))

    accuracy_adj = (y_pred_adj == y_true).mean()
    kappa_adj    = cohen_kappa_score(y_true, y_pred_adj)

    threshold_adjusted_df = pd.DataFrame({
        'class'              : CLASSES,
        'malignant'          : [c in MALIGNANT_CLASSES for c in CLASSES],
        'support'            : [int((y_true == i).sum()) for i in range(NUM_CLASSES)],
        'sensitivity_recall' : recall_adj,
        'specificity'        : specificity_adj,
        'precision'          : precision_adj,
        'f1'                 : f1_adj,
    })
    threshold_adjusted_df.to_csv(METRICS_DIR / f'{tag}_per_class_metrics_threshold_adjusted.csv', index=False)

    print(f"\nArgmax             -> accuracy: {accuracy:.4f}   Cohen's kappa: {kappa:.4f}")
    print(f"Threshold-adjusted -> accuracy: {accuracy_adj:.4f}   Cohen's kappa: {kappa_adj:.4f}")
    print()
    print(f"{'class':28s} {'sens (argmax)':>14s} {'sens (adj)':>12s} {'spec (argmax)':>14s} {'spec (adj)':>12s}")
    for cls in MALIGNANT_CLASSES:
        i = CLASSES.index(cls)
        print(f"{cls:28s} {recall[i]:>14.4f} {recall_adj[i]:>12.4f} {specificity[i]:>14.4f} {specificity_adj[i]:>12.4f}")

    # Comparison against project targets
    comparison_df = pd.DataFrame([
        {'metric': 'Macro AUC (test)',                       'achieved': macro_auc,                'target': 0.93},
        {'metric': 'Melanoma sensitivity (argmax)',          'achieved': recall[melanoma_idx],     'target': 0.85},
        {'metric': 'Melanoma sensitivity (threshold-adj.)',  'achieved': recall_adj[melanoma_idx], 'target': 0.85},
        {'metric': "Cohen's kappa (argmax)",                 'achieved': kappa,                    'target': 0.80},
        {'metric': "Cohen's kappa (threshold-adj.)",         'achieved': kappa_adj,                'target': 0.80},
        {'metric': 'Overall accuracy (argmax)',              'achieved': accuracy,                 'target': None},
        {'metric': 'Overall accuracy (threshold-adj.)',      'achieved': accuracy_adj,             'target': None},
        {'metric': 'Macro F1 (argmax)',                       'achieved': macro_f1,                 'target': None},
        {'metric': 'Balanced accuracy (argmax)',              'achieved': bal_acc,                  'target': None},
        {'metric': "MCC (argmax)",                            'achieved': mcc,                      'target': None},
    ])
    comparison_df['gap'] = comparison_df['target'] - comparison_df['achieved']
    comparison_df.to_csv(METRICS_DIR / f'{tag}_target_comparison.csv', index=False)
    print("\nComparison vs targets:")
    print(comparison_df.round(4).to_string(index=False))

    # Pigmented-lesion confusion cluster (melanoma -> nevus, melanoma -> PBK)
    nevus_idx = CLASSES.index('nevus')
    pbk_idx   = CLASSES.index('pigmented benign keratosis')
    mel_total = cm[melanoma_idx, :].sum()
    mel_to_nevus = cm[melanoma_idx, nevus_idx]
    mel_to_pbk   = cm[melanoma_idx, pbk_idx]

    pigmented_cluster_df = pd.DataFrame([
        {'confusion': 'melanoma -> nevus',                       'count': int(mel_to_nevus), 'melanoma_total': int(mel_total), 'fraction': mel_to_nevus / mel_total if mel_total else float('nan')},
        {'confusion': 'melanoma -> pigmented benign keratosis',  'count': int(mel_to_pbk),   'melanoma_total': int(mel_total), 'fraction': mel_to_pbk / mel_total if mel_total else float('nan')},
    ])
    pigmented_cluster_df.to_csv(METRICS_DIR / f'{tag}_pigmented_cluster_confusion.csv', index=False)
    print("\nPigmented-lesion confusion cluster (melanoma misclassifications):")
    print(pigmented_cluster_df.round(4).to_string(index=False))

    return {
        'macro_auc': macro_auc, 'melanoma_auc': auc_per_class['melanoma'],
        'melanoma_sens_argmax': recall[melanoma_idx], 'melanoma_sens_adj': recall_adj[melanoma_idx],
        'kappa_argmax': kappa, 'kappa_adj': kappa_adj,
        'accuracy_argmax': accuracy, 'accuracy_adj': accuracy_adj,
        'macro_f1': macro_f1, 'balanced_accuracy': bal_acc, 'mcc': mcc,
    }


print("run_full_eval defined ✓")

## 14 - Run Evaluation on val_v6 and test_v6 (untouched 7-class splits)

In [ ]:
val_df  = pd.read_csv(DATA_DIR / 'val_v6.csv')
test_df = pd.read_csv(DATA_DIR / 'test_v6.csv')

val_results  = run_full_eval(val_df,  'val_v8', 'v8 VAL (EfficientNet-B4, equal-class undersampled on v6)')
test_results = run_full_eval(test_df, 'v8',     'v8 TEST (EfficientNet-B4, equal-class undersampled on v6)')

## 15 - Comparison vs v6 (B4, 7-class, nevus-only cap, 4,624 train images)

`v8` is a single-variable change vs. `v6`: the only difference is the
training-set class balance (equal-class undersampling to 518 images,
74/class, vs. v6's nevus-capped 4,624 images). `val_v6.csv`/`test_v6.csv` are
identical for both runs.

In [ ]:
v6_cmp = pd.read_csv(METRICS_DIR / 'test_target_comparison_v6.csv').set_index('metric')['achieved']
v6_pcm = pd.read_csv(METRICS_DIR / 'test_per_class_metrics_v6.csv').set_index('class')

rows = [
    ('Macro AUC',                          'Macro AUC (test)',                      test_results['macro_auc']),
    ('Melanoma AUC',                       None,                                     test_results['melanoma_auc']),
    ('Melanoma sensitivity (argmax)',      'Melanoma sensitivity (argmax)',         test_results['melanoma_sens_argmax']),
    ('Melanoma sensitivity (thresh-adj.)', 'Melanoma sensitivity (threshold-adj.)', test_results['melanoma_sens_adj']),
    ("Cohen's kappa (argmax)",             "Cohen's kappa (argmax)",                test_results['kappa_argmax']),
    ('Overall accuracy (argmax)',          'Overall accuracy (argmax)',             test_results['accuracy_argmax']),
    ('Macro F1',                           'Macro F1 (argmax)',                     test_results['macro_f1']),
    ('Balanced accuracy',                  'Balanced accuracy (argmax)',            test_results['balanced_accuracy']),
    ('MCC',                                'MCC (argmax)',                          test_results['mcc']),
]

comparison_rows = []
for label, cmp_key, v8_val in rows:
    if label == 'Melanoma AUC':
        v6_val = v6_pcm.loc['melanoma', 'auc']
    else:
        v6_val = v6_cmp.get(cmp_key, float('nan'))
    comparison_rows.append({'metric': label, 'v6_7cls_380px_B4_nevuscap': v6_val, 'v8_7cls_380px_B4_equalclass': v8_val,
                             'delta_v8_minus_v6': v8_val - v6_val})

final_comparison = pd.DataFrame(comparison_rows)
final_comparison.to_csv(METRICS_DIR / 'comparison_v6_vs_v8.csv', index=False)
print("="*70)
print("  FINAL COMPARISON: v6 (B4, nevus-only cap, 4624 train) vs v8 (B4, equal-class undersampling, 518 train)")
print("="*70)
print(final_comparison.round(4).to_string(index=False))

---
## Conclusion

Equal-class undersampling (518 train images, 74/class) **regresses every
metric** vs. v6 (4,624 train images, nevus-only cap of 3,000) on both
melanoma-specific and overall-discrimination measures:

- Macro AUC: 0.8789 -> 0.8534 (-0.0255)
- Melanoma AUC: 0.7927 -> 0.7482 (-0.0445)
- Melanoma sensitivity (argmax): 0.4262 -> 0.3080 (-0.1181)
- Melanoma sensitivity (threshold-adjusted): 0.7679 -> 0.7342 (-0.0338, both
  miss the 0.85 target)
- Cohen's kappa: 0.4928 -> 0.4111 (-0.0817)
- Overall accuracy: 0.6259 -> 0.5556 (-0.0704)
- Macro F1: 0.5216 -> 0.4653 (-0.0563)
- Balanced accuracy: 0.5350 -> 0.5257 (-0.0093)
- MCC: 0.4973 -> 0.4156 (-0.0816)

Notably, even balanced accuracy regresses here -- unlike the earlier v7
experiment (EfficientNet-B0, equal-class undersampling on the 8-class
dataset), where balanced accuracy was the *one* metric that improved. This
rules out the hypothesis that v7's balanced-accuracy gain was specific to the
more-imbalanced 8-class taxonomy: even on the milder 7-class v6 distribution,
cutting the training set from 4,624 to 518 images (74/class) is too severe,
and data starvation dominates any balancing benefit -- for both EfficientNet-B0
(v7) and EfficientNet-B4 (v8), and for both the 8-class and 7-class
taxonomies.

**Decision**: `final_model_v4.pth` (EfficientNet-B4, 380px, 8-class) remains
the best overall model; `final_model_v6.pth` remains the best 7-class
(corrected-label) model. Equal-class undersampling to the true minority-class
size (dermatofibroma=74) is conclusively not viable on this dataset,
independent of backbone or taxonomy. v8 is documented as a Phase 8
"future work"/limitations finding alongside v7. All v8 artifacts (notebook,
checkpoints, metrics, figures) preserved at the project root, not archived.